<div style="border-left: 4px solid #27ae60; padding-left: 15px; margin-bottom: 20px;">
  <h1 style="margin-top: 0;">🏭⚡ 03. Exploratory Data Analysis: Net-Load Dynamics & Prescriptive Landscape</h1>
  <p style="font-size: 1.1em; color: #888;">Bringing it all together: matching machine demand with renewable supply and market costs to build a smart scheduling engine.</p>
</div>

### 🎯 Main Goals & Business Value
This is where we move from just predicting the future to actively improving it. We are combining our machine's energy needs with outside electricity prices to build a financial "Digital Twin" of the factory:

*   **⚖️ The Real Power Draw:** Figuring out exactly how much extra power we need to buy from the grid after we've used up all our free solar energy.
*   **💶 The Cost to Beat:** How much does a normal, everyday factory schedule cost? We need to calculate this baseline so our smart scheduling program has a clear financial target to beat.
*   **📐 Setting the Rules:** Defining what the machine *must* do (like taking breaks to cool down or running only during work shifts) so our program doesn't suggest a plan that damages the equipment.

### 🛠️ Core Tools & Approach
*   **Data Engineering:** `Polars` (Used for blazing-fast data processing because this real-world industrial dataset is simply too heavy for standard tools like Pandas), `Numpy`, and `Plotly` (for visual charts).
*   **Business Rules:** Translating real-world factory limits into mathematical rules that a computer can follow.
*   **Scheduling Prep:** Building the exact data foundation for our future optimization program, which will calculate the absolute best and cheapest times to run the machines.

---

### 🗄️ Data Structure
This section brings everything together, simulating the entire factory's behavior in one place.

| Domain | Feature Examples | Purpose |
| :--- | :--- | :--- |
| 🔌 **Power Needed** | `Net_Power_Draw` | What we actually have to buy from the power grid after our free solar energy runs out. |
| 💶 **Daily Costs** | `Current_Hourly_Spend` | The normal, everyday cost of running the factory without any smart scheduling. |
| 🚧 **Safety Rules** | `Operational_State`, `Shift_Hours` | The strict physical and business rules the machines must follow to stay safe and functioning. |

> 
**Next Step Alignment:** This combined data is the launchpad for our smart scheduling program. It gives us the proof we need to show that dynamic scheduling will heavily reduce both factory costs and carbon emissions.

In [1]:
# Standard Library
import datetime
import warnings

# Data Manipulation
import polars as pl

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Notebook Configuration
%matplotlib inline
warnings.filterwarnings('ignore')

# Set Polars config for clean, readable terminal outputs
pl.Config.set_tbl_rows(8)
pl.Config.set_fmt_str_lengths(50)

polars.config.Config

In [2]:
# 🚀 High-Performance Dual Ingestion
import polars as pl
import plotly.express as px

# Define relative paths from the notebooks/EDA/ folder
machine_path = "../../data/processed/features/TEC_48S_final_features.parquet"
solar_path = "../../data/processed/features/IPE_PV_final_features.parquet"

try:
    # Load both datasets into memory
    df_machine = pl.read_parquet(machine_path)
    df_solar = pl.read_parquet(solar_path)
    
    print("✅ Datasets loaded successfully.")
    print("-" * 65)
    print(f"🏭 Factory Machine (TEC_48S): {df_machine.height:,} rows | {df_machine.width} columns")
    print(f"☀️ Solar Environment (IPE_PV): {df_solar.height:,} rows | {df_solar.width} columns")
    print(f"📅 Timeline Sync: {df_machine['WsDateTime'].min().date()} to {df_machine['WsDateTime'].max().date()}")
    print("-" * 65)
    
    # Display a quick verification snapshot
    display(df_machine.select(["WsDateTime", "Angle_U1", "DayAhead_Price_EUR_MWh"]).head(2))
    display(df_solar.select(["WsDateTime", "AC_ActivePower", "Air_Temperature_C"]).head(2))

except FileNotFoundError as e:
    print(f"❌ Error: Could not find the feature matrix. Check your paths: {e}")

✅ Datasets loaded successfully.
-----------------------------------------------------------------
🏭 Factory Machine (TEC_48S): 6,324,301 rows | 182 columns
☀️ Solar Environment (IPE_PV): 1,710,720 rows | 20 columns
📅 Timeline Sync: 2024-01-01 to 2024-12-31
-----------------------------------------------------------------


WsDateTime,Angle_U1,DayAhead_Price_EUR_MWh
datetime[ms],f64,f64
2024-01-01 00:14:56.037,0.0,0.01
2024-01-01 00:15:01.037,0.0,0.01


WsDateTime,AC_ActivePower,Air_Temperature_C
datetime[ms],f64,f64
2024-01-01 00:00:00,0.0,7.0
2024-01-01 00:00:05,0.0,7.0


In [3]:
# 🚀 High-Performance Dual Integrity Diagnostic
import polars as pl

print("=" * 75)
print("🧹 COMPREHENSIVE PIPELINE & INTEGRITY DIAGNOSTIC: UNIFIED PLANT")
print("=" * 75)

# Dictionary to iterate through both datasets cleanly
datasets = {
    "🏭 Factory Machine (TEC_48S)": df_machine, 
    "☀️ Solar Environment (IPE_PV)": df_solar
}

for name, df in datasets.items():
    print(f"\nEvaluating: {name}")
    print("-" * 60)
    
    # 1. Structural Integrity (Nulls & Duplicates)
    null_counts = df.select(pl.all().null_count())
    missing_data = (
        null_counts
        .unpivot(variable_name="Feature", value_name="Missing_Count")
        .filter(pl.col("Missing_Count") > 0)
    )
    missing_passed = missing_data.height == 0

    duplicate_count = df.filter(pl.col("WsDateTime").is_duplicated()).height
    dup_passed = duplicate_count == 0

    print(f"{'✅' if missing_passed else '⚠️'} Null Check:      { 'Zero missing values.' if missing_passed else f'{missing_data.height} columns have missing values.' }")
    print(f"{'✅' if dup_passed else '⚠️'} Duplicate Check: { 'Perfectly unique temporal grid.' if dup_passed else f'{duplicate_count:,} duplicate timestamps found.' }")

    # 2. Temporal Continuity & Coverage
    start_time = df.select(pl.col("WsDateTime").min()).item()
    end_time = df.select(pl.col("WsDateTime").max()).item()
    max_gap_s = df.select(pl.col("WsDateTime").diff().dt.total_seconds().max()).item()
    max_gap_hrs = (max_gap_s / 3600.0) if max_gap_s is not None else 0

    print(f"✅ Time Coverage:   {start_time.date()} to {end_time.date()} ({(end_time - start_time).days} days)")
    if max_gap_hrs > 24:
        print(f"⚠️ Gap Warning:     Longest telemetry dropout is {max_gap_hrs:.1f} hours.")
    else:
        print(f"✅ Continuity Check:No major multi-day sensor dropouts (Max gap: {max_gap_hrs:.1f} hours).")

    # 3. Physical Boundary Checks (No negative generation/loads)
    physical_keywords = ["power", "current", "voltage", "yield", "p1", "p2", "p3", "p_total"]
    physical_cols = [
        c for c in df.columns 
        if any(kw in c.lower() for kw in physical_keywords) 
        and df[c].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64]
    ]

    if physical_cols:
        min_exprs = [pl.col(c).min().alias(c) for c in physical_cols]
        min_values = df.select(min_exprs).row(0)
        negative_issues = [physical_cols[i] for i, val in enumerate(min_values) if val is not None and val < 0]

        if not negative_issues:
            print(f"✅ Physics Check:   Passed. All {len(physical_cols)} physical sensors show valid positive ranges.")
        else:
            print(f"⚠️ Physics Warning: Negative values detected in {len(negative_issues)} sensors: {negative_issues[:3]}...")

print("\n" + "=" * 75)
print("🚀 PIPELINE STATUS: PRODUCTION-READY")
print("=" * 75)

🧹 COMPREHENSIVE PIPELINE & INTEGRITY DIAGNOSTIC: UNIFIED PLANT

Evaluating: 🏭 Factory Machine (TEC_48S)
------------------------------------------------------------
✅ Null Check:      Zero missing values.
✅ Duplicate Check: Perfectly unique temporal grid.
✅ Time Coverage:   2024-01-01 to 2024-12-31 (365 days)
✅ Continuity Check:No major multi-day sensor dropouts (Max gap: 0.0 hours).
✅ Physics Check:   Passed. All 4 physical sensors show valid positive ranges.

Evaluating: ☀️ Solar Environment (IPE_PV)
------------------------------------------------------------
✅ Null Check:      Zero missing values.
✅ Duplicate Check: Perfectly unique temporal grid.
✅ Time Coverage:   2024-01-01 to 2024-04-08 (98 days)
✅ Continuity Check:No major multi-day sensor dropouts (Max gap: 0.0 hours).
✅ Physics Check:   Passed. All 12 physical sensors show valid positive ranges.

🚀 PIPELINE STATUS: PRODUCTION-READY
